<a href="https://colab.research.google.com/github/SuperDataWorld/Python/blob/main/Pulling_Tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["X_BEARER_TOKEN"] = "AAAAAAAAAAAAAAAAAAAAADU18gEAAAAAcHHQS8ZfUJUSfnsCqC8v8jQckYY%3DAeeC0bC5N0JoYQ3A9vF9OKyAzZuDa8f23XCi7Eik2rFAB9B3Bo"

In [2]:
token = os.getenv("X_BEARER_TOKEN")

In [ ]:
import os
import time
import requests
import pandas as pd

BASE_URL = "https://api.x.com/2/tweets/search/recent"  # Search recent Posts endpoint :contentReference[oaicite:1]{index=1}

def search_recent_posts_to_df(
    bearer_token: str,
    query: str,
    max_pages: int = 5,
    max_results: int = 100,
    sleep_seconds: float = 1.0,
) -> pd.DataFrame:
    """
    Descarga posts (últimos 7 días) que matcheen query, con paginación (next_token),
    y devuelve un DataFrame.
    """
    headers = {"Authorization": f"Bearer {bearer_token}"}  # Bearer auth header :contentReference[oaicite:2]{index=2}

    # Campos recomendados (ajusta a tu necesidad)
    params = {
        "query": query,                 # required :contentReference[oaicite:3]{index=3}
        "max_results": max_results,      # 10..100 :contentReference[oaicite:4]{index=4}
        "tweet.fields": ",".join([       # tweet.fields list :contentReference[oaicite:5]{index=5}
            "id", "text", "created_at", "author_id", "lang", "geo", "public_metrics"
        ]),
        "expansions": ",".join([
            "author_id",
            "geo.place_id"
        ]),
        "user.fields": ",".join([
            "id", "name", "username", "created_at"
        ]),
        "place.fields": ",".join([      # place.fields list :contentReference[oaicite:6]{index=6}
            "id", "full_name", "country", "country_code", "place_type", "geo"
        ]),
        "sort_order": "recency",         # option :contentReference[oaicite:7]{index=7}
    }

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token  # pagination token :contentReference[oaicite:8]{index=8}
        else:
            params.pop("next_token", None)

        r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()

        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}

        # Indexa "includes" para hacer join fácil
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": (t.get("public_metrics") or {}).get("retweet_count"),
                "reply_count": (t.get("public_metrics") or {}).get("reply_count"),
                "like_count": (t.get("public_metrics") or {}).get("like_count"),
                "quote_count": (t.get("public_metrics") or {}).get("quote_count"),
            })

        next_token = meta.get("next_token")
        if not next_token:
            break

        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


if __name__ == "__main__":
    # 1) Pon tu token en variable de entorno:
    # export X_BEARER_TOKEN="..."
    token = os.getenv("X_BEARER_TOKEN")
    if not token:
        raise ValueError("Falta X_BEARER_TOKEN en variables de entorno.")

    # 2) Query: "scam" + EE.UU. (si hay geo), excluyendo retweets
    #    Ojo: place_country:US depende de que el tweet tenga place/geo.
    
    #hemos cambiado el query para que sea más específico, incluye más palabras que no sean solo scam (y por lo tanto otros tipos de scam) pero también excluye 
    #tweets en los que las personas pueden usar la palabras de la lista pero no tienen que ver con el tipo de scam que estamos buscando
   
    query = '''
(estafa OR fraude OR "fraude financiero" OR "fraude electrónico" 
OR "estafa electrónica" OR "transferencias fraudulentas" 
OR "fraude bancario" OR "estafa de inversión" 
OR "esquema ponzi" OR malversación OR "fraude de valores" OR phishing)  
place_country: DO
-is:retweet 
-videojuegos -deportes -fútbol -meme -política -gobierno
'''

#hemos cambiado el max_pages de 10 a 100 para que el resultado sean 10.000 tweets 
    df = search_recent_posts_to_df(
        bearer_token=token,
        query=query,
        max_pages=100,
        max_results=100
    )

    # 3) Guardar el resultado
    df.to_csv("scam_us_recent_posts.csv", index=False, encoding="utf-8")
    # o mejor para análisis: df.to_parquet("scam_us_recent_posts.parquet", index=False)

    print(df.head())
    print(f"Total filas: {len(df)}")

RuntimeError: X API error 402: {"account_id":2028447877882867712,"title":"CreditsDepleted","detail":"Your enrolled account [2028447877882867712] does not have any credits to fulfill this request.","type":"https://api.twitter.com/2/problems/credits"}

In [9]:
df.to_csv("scam_us_recent_posts.csv", index=False, encoding="utf-8")
print(df.head())
print(f"Total filas: {len(df)}")

NameError: name 'df' is not defined

In [4]:
"""
❗ Realidad en 2026

La API de X ya no es verdaderamente gratuita como antes.
El plan free:

Tiene límites muy estrictos

O directamente no permite el endpoint search/recent

Y cuando se acaban los créditos → error 402

“All developers” no significa gratis ilimitado.
Significa que el endpoint está disponible para todos los planes… pero dentro de los créditos de tu plan.
"""


'\n❗ Realidad en 2026\n\nLa API de X ya no es verdaderamente gratuita como antes.\nEl plan free:\n\nTiene límites muy estrictos\n\nO directamente no permite el endpoint search/recent\n\nY cuando se acaban los créditos → error 402\n\n“All developers” no significa gratis ilimitado.\nSignifica que el endpoint está disponible para todos los planes… pero dentro de los créditos de tu plan.\n'

In [5]:
tweets_df

NameError: name 'tweets_df' is not defined

In [6]:
tweets[0]

NameError: name 'tweets' is not defined

In [7]:
tweets_df.to_csv('tweets.csv') 
files.download('tweets.csv')

NameError: name 'tweets_df' is not defined